In [1]:
import pandas as pd

In [2]:
df_raw = pd.read_csv("prices_round_2_day_0.csv", header=None, names=["raw"])

In [8]:
df = df_raw["raw"].str.split(";", expand=True)
columns = [
    "day", "timestamp", "product",
    "bid_price_1", "bid_volume_1",
    "bid_price_2", "bid_volume_2",
    "bid_price_3", "bid_volume_3",
    "ask_price_1", "ask_volume_1",
    "ask_price_2", "ask_volume_2",
    "ask_price_3", "ask_volume_3",
    "mid_price", "profit_and_loss"
]
df.columns = columns[:df.shape[1]]

In [4]:
import numpy as np
import pandas as pd

# 1. Convert all numeric columns properly
price_cols = ["bid_price_1", "bid_price_2", "bid_price_3", "ask_price_1", "ask_price_2", "ask_price_3"]
volume_cols = ["bid_volume_1", "bid_volume_2", "bid_volume_3", "ask_volume_1", "ask_volume_2", "ask_volume_3"]
other_cols = ["mid_price", "profit_and_loss"]
df[price_cols + volume_cols+other_cols] = df[price_cols + volume_cols+other_cols].apply(pd.to_numeric, errors='coerce')

# 2. Replace NaNs with large/small values to safely use argmin/argmax
bid_price_array = df[["bid_price_1", "bid_price_2", "bid_price_3"]].fillna(-np.inf).to_numpy()
bid_volume_array = df[["bid_volume_1", "bid_volume_2", "bid_volume_3"]].to_numpy()
best_bid_idx = bid_price_array.argmax(axis=1)
df["best_bid_price"] = bid_price_array[np.arange(len(df)), best_bid_idx]
df["best_bid_volume"] = bid_volume_array[np.arange(len(df)), best_bid_idx]

ask_price_array = df[["ask_price_1", "ask_price_2", "ask_price_3"]].fillna(np.inf).to_numpy()
ask_volume_array = df[["ask_volume_1", "ask_volume_2", "ask_volume_3"]].to_numpy()
best_ask_idx = ask_price_array.argmin(axis=1)
df["best_ask_price"] = ask_price_array[np.arange(len(df)), best_ask_idx]
df["best_ask_volume"] = ask_volume_array[np.arange(len(df)), best_ask_idx]

# 3. Compute bid-ask spread
df["bid_ask_spread"] = df["best_ask_price"] - df["best_bid_price"]
df=df[1:]
# 4. Add cumulative profit and loss column
df["current_portfolio"] = df["profit_and_loss"].cumsum()

In [9]:
df.drop(columns=[ "bid_price_2", "bid_price_3","ask_price_3","ask_price_2","bid_volume_2", "bid_volume_3","ask_volume_2", "ask_volume_3"], inplace=True)
df

,day,timestamp,product,bid_price_1,bid_volume_1,ask_price_1,ask_volume_1,mid_price,profit_and_loss
0,day,timestamp,product,bid_price_1,bid_volume_1,ask_price_1,ask_volume_1,mid_price,profit_and_loss
1,0,0,RAINFOREST_RESIN,9992,30,10008,30,10000.0,0.0
2,0,0,DJEMBES,13493,72,13494,72,13493.5,0.0
3,0,0,CROISSANTS,4321,111,4322,111,4321.5,0.0
4,0,0,JAMS,6631,210,6633,210,6632.0,0.0
...,...,...,...,...,...,...,...,...,...
79996,0,999900,PICNIC_BASKET2,30255,2,30260,18,30257.5,0.0
79997,0,999900,CROISSANTS,4275,124,4276,124,4275.5,0.0
79998,0,999900,JAMS,6541,183,6543,183,6542.0,0.0
79999,0,999900,DJEMBES,13409,64,13410,64,13409.5,0.0
